## Example 1: Write a shard with TarWriter and read it back

In [ ]:
import tempfile, os
import webdataset as wds

shard = os.path.join(tempfile.mkdtemp(), "shard-000000.tar")
with wds.TarWriter(shard) as sink:
    for i in range(5):
        sink.write({"__key__": f"s{i:03d}", "txt": f"hello {i}", "cls": i})
samples = list(wds.WebDataset(shard, shardshuffle=False))
assert [s["__key__"] for s in samples] == [f"s{i:03d}" for i in range(5)]
assert samples[2]["txt"] == b"hello 2", samples[2]["txt"]
print(len(samples), "samples:", sorted(samples[0]))


## Example 2: Decode fields to Python objects

In [ ]:
import tempfile, os
import webdataset as wds

shard = os.path.join(tempfile.mkdtemp(), "data.tar")
with wds.TarWriter(shard) as sink:
    sink.write({"__key__": "a", "json": {"x": 1, "tags": ["p", "q"]}, "cls": 7})
(sample,) = wds.WebDataset(shard, shardshuffle=False).decode()
assert sample["json"] == {"x": 1, "tags": ["p", "q"]}, sample["json"]
assert sample["cls"] == 7
print(sample["json"], sample["cls"])


## Example 3: Several shards with ShardWriter and a brace pattern

In [ ]:
import tempfile, os
import webdataset as wds

root = tempfile.mkdtemp()
with wds.ShardWriter(os.path.join(root, "part-%06d.tar"), maxcount=4) as sink:
    for i in range(10):
        sink.write({"__key__": f"k{i}", "txt": str(i)})
shards = sorted(os.listdir(root))
assert shards == ["part-000000.tar", "part-000001.tar", "part-000002.tar"], shards
url = os.path.join(root, "part-{000000..000002}.tar")
keys = sorted(s["__key__"] for s in wds.WebDataset(url, shardshuffle=False))
assert keys == sorted(f"k{i}" for i in range(10)), keys
print(shards, len(keys))


## Example 4: A pipeline: decode, map, tuple and batch

In [ ]:
import tempfile, os
import webdataset as wds

shard = os.path.join(tempfile.mkdtemp(), "p.tar")
with wds.TarWriter(shard) as sink:
    for i in range(6):
        sink.write({"__key__": f"{i}", "txt": f"v{i}", "cls": i})
ds = (wds.WebDataset(shard, shardshuffle=False).decode()
      .to_tuple("txt", "cls").map_tuple(str.upper, lambda c: c * 10).batched(3))
batches = list(ds)
assert len(batches) == 2, batches
assert list(batches[0][0]) == ["V0", "V1", "V2"] and list(batches[0][1]) == [0, 10, 20], batches[0]
print(batches)


## Example 5: NumPy arrays round-trip

In [ ]:
import tempfile, os
import numpy as np
import webdataset as wds

shard = os.path.join(tempfile.mkdtemp(), "n.tar")
arr = np.arange(12, dtype=np.float32).reshape(3, 4)
with wds.TarWriter(shard) as sink:
    sink.write({"__key__": "m", "npy": arr})
(sample,) = wds.WebDataset(shard, shardshuffle=False).decode()
assert np.array_equal(sample["npy"], arr) and sample["npy"].dtype == np.float32
print(sample["npy"].shape, sample["npy"].dtype)
